# V2 — Binoculars: generacja + ewaluacja (Colab)

Pipeline dla `binoculars` w **dwóch osobnych procesach** (mniej OOM):

1. generacja baseline + stego (Qwen / Llama / Gemma) → subprocess kończy się, GPU wolne
2. Binoculars eval (Falcon 4-bit/8-bit) → osobny subprocess

**Colab T4 (16 GB):** powinno działać po fixie kwantyzacji Falcon. Jeśli nadal OOM → użyj `Kaggle_Binoculars_Generate_Eval.ipynb` (więcej RAM systemowego).

Wyniki na Drive: `runs/<run>/evaluation/binoculars_results.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Steganography_benchmarks_V2')
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
RUNS_DIR = PROJECT_DIR / 'runs'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)

print('PROJECT_DIR:', PROJECT_DIR)
print('RUNS_DIR:', RUNS_DIR)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "bitsandbytes>=0.43.1" "accelerate>=0.33.0"

import bitsandbytes  # noqa: F401 — musi być załadowany przed from_pretrained
print('bitsandbytes OK')

In [ ]:
from getpass import getpass
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN (Llama/Gemma): ')

In [ ]:
# --- KONFIGURACJA ---
MODELS = ['qwen', 'llama', 'gemma']
THRESHOLDS = [0.0, 0.01, 0.05, 0.1]

TOP_N = 16
MAX_NEW_TOKENS = 512
SKIP_IF_EVALUATED = True
CONTINUE_ON_ERROR = True

import gc
import json
import sys
from pathlib import Path

import torch
from notebook_runner import run_live

TEST = 'binoculars'
PLATFORM = 'colab'


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def _th_match(a: float, b: float) -> bool:
    return abs(float(a) - float(b)) < 1e-9


def find_run(runs_dir: Path, model: str, threshold: float) -> Path | None:
    if not runs_dir.exists():
        return None
    candidates: list[tuple[str, Path]] = []
    for manifest_path in runs_dir.glob('*/manifest.json'):
        try:
            m = json.loads(manifest_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            continue
        if m.get('test') != TEST:
            continue
        if m.get('model_key') != model:
            continue
        if not _th_match(m.get('threshold', -1), threshold):
            continue
        if m.get('status') != 'completed':
            continue
        candidates.append((m.get('updated_at', ''), manifest_path.parent))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def has_eval(run_dir: Path) -> bool:
    return (run_dir / 'evaluation' / 'binoculars_results.json').is_file()


def build_gen_cmd(model: str, threshold: float, run_dir: Path | None = None) -> list[str]:
    cmd = [
        sys.executable, 'generate_responses.py',
        '--test', TEST,
        '--threshold', str(threshold),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', PLATFORM,
        '--output-root', 'runs',
    ]
    if run_dir is not None:
        cmd += ['--run-dir', str(run_dir)]
    else:
        cmd += ['--model', model]
    return cmd


def build_eval_cmd(run_dir: Path) -> list[str]:
    return [
        sys.executable, 'evaluate_responses.py',
        '--run-dir', str(run_dir),
        '--platform', PLATFORM,
    ]


jobs = [(m, th) for m in MODELS for th in THRESHOLDS]
print(f'Plan: {len(jobs)} runów (gen subprocess → eval subprocess)')
print(f'  modele: {MODELS}')
print(f'  progi: {THRESHOLDS}')

ok, skipped, failed = [], [], []

for index, (model, threshold) in enumerate(jobs, start=1):
    label = f'{model} | th={threshold}'
    print('\n' + '=' * 60)
    print(f'[{index}/{len(jobs)}] {label}')
    print('=' * 60)

    free_gpu()
    run_dir = find_run(RUNS_DIR, model, threshold)

    if SKIP_IF_EVALUATED and run_dir is not None and has_eval(run_dir):
        print(f'Pomijam — już ewaluowany: {run_dir.name}')
        skipped.append(label)
        continue

    try:
        if run_dir is None:
            print('[1/2] Generacja (nowy run)...')
            run_live(build_gen_cmd(model, threshold))
            run_dir = find_run(RUNS_DIR, model, threshold)
            if run_dir is None:
                raise RuntimeError('Brak run dir po generacji')
        else:
            print(f'[1/2] Generacja pominięta — RAW OK: {run_dir.name}')

        free_gpu()
        print('[2/2] Binoculars eval (osobny proces, Falcon 4-bit/8-bit)...')
        run_live(build_eval_cmd(run_dir))

        if not has_eval(run_dir):
            raise RuntimeError('Brak binoculars_results.json')

        data = json.loads((run_dir / 'evaluation' / 'binoculars_results.json').read_text())
        ok.append(label)
        print(
            f"OK: baseline={data['baseline_binoculars_score']:.3f}, "
            f"stego={data['binoculars_score']:.3f}, "
            f"delta={data['binoculars_score_delta']:+.3f}, "
            f"AI rate={data['ai_detection_rate']:.1%}"
        )
    except Exception as exc:
        print(f'BŁĄD: {exc}')
        failed.append((label, str(exc)))
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        free_gpu()

print('\n' + '#' * 60)
print('PODSUMOWANIE')
print('#' * 60)
print(f'OK: {len(ok)} | Pominięte: {len(skipped)} | Błędy: {len(failed)}')
if failed:
    print('\nBłędne runy:')
    for label, err in failed:
        print(f'  - {label}: {err}')

In [ ]:
# Tabela wyników binoculars
import json
from pathlib import Path

rows = []
for manifest_path in sorted(Path('runs').glob('*/manifest.json')):
    m = json.loads(manifest_path.read_text(encoding='utf-8'))
    if m.get('test') != 'binoculars':
        continue
    run_dir = manifest_path.parent
    eval_path = run_dir / 'evaluation' / 'binoculars_results.json'
    row = {
        'model': m.get('model_key'),
        'th': m.get('threshold'),
        'status': m.get('status'),
        'run': run_dir.name,
        'eval': eval_path.is_file(),
    }
    if eval_path.is_file():
        d = json.loads(eval_path.read_text())
        row['baseline'] = d.get('baseline_binoculars_score')
        row['stego'] = d.get('binoculars_score')
        row['delta'] = d.get('binoculars_score_delta')
        row['ai_rate'] = d.get('ai_detection_rate')
    rows.append(row)

rows.sort(key=lambda r: (r['model'], float(r['th'])))
print(f'Runów binoculars: {len(rows)}\n')
print(f"{'model':<6} {'th':<5} {'gen':<12} {'eval':<5} {'baseline':>9} {'stego':>9} {'delta':>9} {'AI%':>7}")
print('-' * 72)
for r in rows:
    if r.get('eval'):
        print(
            f"{r['model']:<6} {r['th']:<5} {r['status']:<12} {'yes':<5} "
            f"{r['baseline']:9.3f} {r['stego']:9.3f} {r['delta']:+9.3f} {100*r['ai_rate']:6.1f}%"
        )
    else:
        print(f"{r['model']:<6} {r['th']:<5} {r['status']:<12} {'no':<5}")